In [1]:
import os
import pandas as pd

In [2]:
OVERLAPS = "RDV_MMS_all_overlaps.csv"

In [3]:
#a list of genes curated from scientific literature on PTCL, NOS
PTCL_LITERATURE_GENES = [
    #cell cycle/ tumour suppressors
    "TP53", "CDKN2A", "RB1", "ATM", "TNFRSF14",
    #epigenetic regulators
    "TET2", "DNMT3A", "IDH2", "KMT2D", "SETD2", "SETDB1", "EZH2",
    #TCR/signalling
    "RHOA", "PLCG1", "FYN", "VAV1", "CD28", "CTLA4", "PIK3CA", "AKT1", "AKT2",
    "STAT3", "STAT5B", "JAK1", "JAK3", "IRF4", "PRKCB",
    #TFH/TF
    "GATA3", "TBX21", "BCL11B", "ZEB1", "MYC", "BCL2",
    #apoptosis
    "CARD11", "BCL10", "FAS", "TNFRSF1A",
    #others
    "MTOR", "PRDM2", "PRDM16", "LATS2", "RPL22", "TCL1A", "NOTCH1", "NOTCH2"
]

In [4]:
df = pd.read_csv(OVERLAPS)

In [6]:
#explode to one gene per row
df["gene_name"] = (
    df["gene_name"].astype(str)
      .str.strip("[]")
      .str.replace("'", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.split()
)
df = df.explode("gene_name")

In [7]:
#extract patient and bin
df["patient"] = df["sample"].str.extract(r"^(RDV|MM|MMS)")[0].replace({"MM": "MMS"})
df["bin"] = df["sample"].str.extract(r"(10kb|50kb|500kb|1000kb)")[0]
#uppercase geen symbols for better match
df["gene_name"] = df["gene_name"].astype(str).str.upper().str.strip()

In [18]:
#filter
hits = df[df["gene_name"].isin(PTCL_LITERATURE_GENES)].copy()
hits = hits[["patient", "chr", "gene_name", "event", "bin"]].sort_values(
    ["patient", "chr", "gene_name", "event", "bin"]
)

In [19]:
hits.to_csv("ptcl_literature_hits.tsv", sep="\t", index=False)